# 🌏 Savannakhet GNN — Hybrid Recommendation System
**วิทยานิพนธ์**: ระบบแนะนำสถานที่ท่องเที่ยวจังหวัดสะหวันนะเขต  
**Model**: HeteroGraphSAGE + Content-Based Filtering  
**นักศึกษา**: Phoutthasinh Xaysongkham

---
## 🔀 เลือกโหมดข้อมูล (Cell 2)
| โหมด | ข้อมูล | เหมาะสำหรับ |
|------|--------|-------------|
| `USE_REAL_DATA = True` | ข้อมูลจริงจาก DB (58 users, 71 places) | **นำเสนออาจารย์** |
| `USE_REAL_DATA = False` | Synthetic (10 users, 20 places) | ทดสอบเร็ว |

```
Input ──► HeteroGraphSAGE (K=2, hidden=64)
       ──► GNN Score (60%) + Content-Based (40%)
       ──► Hybrid Fusion ──► Top-K Recommendations
```

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1: ติดตั้ง PyTorch Geometric                       ║
# ╚══════════════════════════════════════════════════════════╝
import subprocess, sys, torch

print('Installing torch_geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu121' if torch.cuda.is_available() else 'cpu'
for pkg in ['torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', pkg, '-q',
            '-f', f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
        ], check=True)
    except Exception as e:
        print(f'Skip {pkg}: {e}')

print(f'[OK] Done! PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2: เลือกโหมด — จริง หรือ Synthetic                ║
# ╚══════════════════════════════════════════════════════════╝

# ════════════════════════════════════════════
#  ตั้งค่าตรงนี้!
USE_REAL_DATA = True   # True = ข้อมูลจริง | False = Synthetic
# ════════════════════════════════════════════

import json, os, random
import numpy as np

CATEGORIES = ['nature', 'culture', 'restaurant', 'hotel',
              'shopping', 'nightlife', 'cafe', 'local_food', 'chill', 'landmark']

if USE_REAL_DATA:
    # ── โหลดข้อมูลจริงจากไฟล์ JSON ──
    from google.colab import files
    print('[*] กรุณาเลือกไฟล์: savannakhet_real_data.json')
    print('    (ไฟล์อยู่ที่ backend/docs/savannakhet_real_data.json)')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    with open(fname, 'r', encoding='utf-8') as f:
        DB = json.load(f)
    CATEGORIES   = DB['categories']
    USERS_DATA   = DB['users']
    PLACES_DATA  = DB['places']
    INTERACTIONS = DB['interactions']
    print(f'[OK] ข้อมูลจริง: {len(USERS_DATA)} users | {len(PLACES_DATA)} places | {len(INTERACTIONS)} interactions')
else:
    # ── ข้อมูล Synthetic จำลอง ──
    PLACES_DATA = [
        {'id':1,  'name':'ວັດສະຫວັນ',              'category':'culture',    'rating':4.8},
        {'id':2,  'name':'ຕະຫຼາດເຊົ້າ',              'category':'local_food', 'rating':4.3},
        {'id':3,  'name':'ຫາດທ່າຍ ຟອງ',            'category':'nature',     'rating':4.6},
        {'id':4,  'name':'ພິພິດທະພັນ ຈ.ສວນ',        'category':'culture',    'rating':4.1},
        {'id':5,  'name':'ຮ້ານອາຫານ ຣິມໂຂງ',        'category':'restaurant', 'rating':4.5},
        {'id':6,  'name':'ສວນສາທາລະນະ',             'category':'chill',      'rating':4.0},
        {'id':7,  'name':'ຄາເຟ ວິວແມ່ນ້ໍາ',          'category':'cafe',       'rating':4.7},
        {'id':8,  'name':'ຕະຫຼາດດິນດ່ວງ',            'category':'shopping',   'rating':4.2},
        {'id':9,  'name':'ບາຣ຺ ຣິມນ້ໍາ',             'category':'nightlife',  'rating':3.9},
        {'id':10, 'name':'ໂຮງແຮມ ສວນ',              'category':'hotel',      'rating':4.4},
        {'id':11, 'name':'ນ້ໍາຕົກ ບົວລະບັດ',          'category':'nature',     'rating':4.9},
        {'id':12, 'name':'ໝູ່ບ້ານ ວັດທະນາທຳ',        'category':'culture',    'rating':4.3},
        {'id':13, 'name':'ຮ້ານເຂົ້າໜຽວ',             'category':'local_food', 'rating':4.6},
        {'id':14, 'name':'ສ​ະ​ພານ ​ມິດ​ຕະ​ພາ​ດ 2',   'category':'landmark',   'rating':4.5},
        {'id':15, 'name':'ຕະຫຼາດ ກາງຄືນ',            'category':'nightlife',  'rating':4.1},
        {'id':16, 'name':'ຮ້ານກາເຟ ລາວ',            'category':'cafe',       'rating':4.4},
        {'id':17, 'name':'ອຸທະຍານ ແຫ່ງຊາດ',          'category':'nature',     'rating':4.7},
        {'id':18, 'name':'ຕະຫຼາດສົດ',               'category':'shopping',   'rating':3.8},
        {'id':19, 'name':'ເຮືອນພັກ ຣິມໂຂງ',          'category':'hotel',      'rating':4.2},
        {'id':20, 'name':'ລານ ວັດທະນາທຳ',            'category':'chill',      'rating':4.0},
    ]
    USERS_DATA = [
        {'id':1, 'username':'dee',  'preferences':['nature','culture','landmark']},
        {'id':2, 'username':'big',  'preferences':['restaurant','cafe','local_food']},
        {'id':3, 'username':'noi',  'preferences':['nightlife','shopping','cafe']},
        {'id':4, 'username':'kai',  'preferences':['culture','chill','local_food']},
        {'id':5, 'username':'pam',  'preferences':['nature','chill','hotel']},
        {'id':6, 'username':'tong', 'preferences':['restaurant','nightlife','shopping']},
        {'id':7, 'username':'may',  'preferences':['culture','landmark','local_food']},
        {'id':8, 'username':'oat',  'preferences':['nature','cafe','chill']},
        {'id':9, 'username':'bee',  'preferences':['hotel','restaurant','cafe']},
        {'id':10,'username':'nam',  'preferences':['shopping','nightlife','landmark']},
    ]
    # สร้าง interactions จาก preferences
    random.seed(42)
    INTERACTIONS = []
    for u in USERS_DATA:
        for p in PLACES_DATA:
            if p['category'] in u['preferences']:
                INTERACTIONS.append({'user_id':u['id'], 'place_id':p['id'], 'action':'view', 'weight':1.0})
            elif random.random() < 0.08:
                INTERACTIONS.append({'user_id':u['id'], 'place_id':p['id'], 'action':'view', 'weight':1.0})
    print(f'[OK] Synthetic: {len(USERS_DATA)} users | {len(PLACES_DATA)} places | {len(INTERACTIONS)} interactions')

# สร้าง index mapping
user_id2idx  = {u['id']: i for i, u in enumerate(USERS_DATA)}
place_id2idx = {p['id']: i for i, p in enumerate(PLACES_DATA)}
place_idx2id = {i: p['id'] for i, p in enumerate(PLACES_DATA)}
NUM_USERS, NUM_PLACES = len(USERS_DATA), len(PLACES_DATA)

# username field compat (real data ใช้ 'username', synthetic ใช้ 'name')
for u in USERS_DATA:
    if 'name' in u and 'username' not in u:
        u['username'] = u['name']

print(f'Mode: {"REAL DATA" if USE_REAL_DATA else "SYNTHETIC"} | Users: {NUM_USERS} | Places: {NUM_PLACES}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3: Import & Setup                                  ║
# ╚══════════════════════════════════════════════════════════╝
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.data import HeteroData
from torch_geometric.utils import negative_sampling
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[OK] Device: {device}')
if torch.cuda.is_available():
    print(f'     GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4: สร้าง HeteroGraph                               ║
# ╚══════════════════════════════════════════════════════════╝

def build_graph():
    data = HeteroData()

    # User Features: Multi-hot (10 dim)
    data['user'].x = torch.tensor(
        [[1.0 if cat in (u.get('preferences') or []) else 0.0 for cat in CATEGORIES]
         for u in USERS_DATA], dtype=torch.float
    )

    # Place Features: One-hot category (10 dim) + rating (1 dim) = 11 dim
    data['place'].x = torch.tensor(
        [[1.0 if cat == p.get('category','') else 0.0 for cat in CATEGORIES]
         + [min(float(p.get('rating', 0)) / 5.0, 1.0)]
         for p in PLACES_DATA], dtype=torch.float
    )

    # Edges จาก Interactions
    edge_src, edge_dst, added = [], [], set()
    for intr in INTERACTIONS:
        uid = intr['user_id']
        pid = intr['place_id']
        if uid not in user_id2idx or pid not in place_id2idx:
            continue
        key = (user_id2idx[uid], place_id2idx[pid])
        if key not in added:
            added.add(key)
            edge_src.append(key[0])
            edge_dst.append(key[1])

    # Fallback: ถ้า edges น้อยเกินไป
    if len(edge_src) < NUM_USERS * 2:
        print(f'[!] Edges น้อย ({len(edge_src)}) — เพิ่ม preference edges')
        for u_idx, u in enumerate(USERS_DATA):
            prefs = u.get('preferences', [])
            for p_idx, p in enumerate(PLACES_DATA):
                if p.get('category') in prefs and (u_idx, p_idx) not in added:
                    added.add((u_idx, p_idx))
                    edge_src.append(u_idx)
                    edge_dst.append(p_idx)

    data['user', 'interacts_with', 'place'].edge_index = torch.tensor(
        [edge_src, edge_dst], dtype=torch.long
    )
    return data

data = build_graph().to(device)
print(f'[OK] Graph: user{data["user"].x.shape} | place{data["place"].x.shape} | edges:{data["user","interacts_with","place"].edge_index.shape[1]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5: HeteroGraphSAGE Model                           ║
# ╚══════════════════════════════════════════════════════════╝

class BaseGNN(torch.nn.Module):
    def __init__(self, hidden_channels=64):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model     = to_hetero(BaseGNN(hidden_channels=64), metadata=data.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(f'[OK] Model: HeteroGraphSAGE (K=2, hidden=64) | Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: เทรนโมเดล (100 Epochs)                         ║
# ╚══════════════════════════════════════════════════════════╝

EPOCHS = 100
losses, accuracies = [], []
pos_edge_index = data['user', 'interacts_with', 'place'].edge_index
n_steps = max(1, pos_edge_index.size(1) // 5)

print('=' * 65)
print(f'  Training HeteroGraphSAGE — {"Real" if USE_REAL_DATA else "Synthetic"} Data')
print(f'  {NUM_USERS} users | {NUM_PLACES} places | {pos_edge_index.shape[1]} edges | {EPOCHS} epochs')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)
    ps, pd  = pos_edge_index[0], pos_edge_index[1]
    pos_out = (out['user'][ps] * out['place'][pd]).sum(dim=-1)

    neg_ei  = negative_sampling(
        edge_index=pos_edge_index,
        num_nodes=(data['user'].num_nodes, data['place'].num_nodes),
        num_neg_samples=pos_edge_index.size(1)
    )
    neg_out = (out['user'][neg_ei[0]] * out['place'][neg_ei[1]]).sum(dim=-1)

    labels = torch.cat([torch.ones(pos_out.size(0)), torch.zeros(neg_out.size(0))]).to(device)
    loss   = F.binary_cross_entropy_with_logits(torch.cat([pos_out, neg_out]), labels)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    with torch.no_grad():
        correct = (pos_out >= 0).float().sum() + (neg_out < 0).float().sum()
        acc = correct.item() / (pos_out.size(0) + neg_out.size(0))
        accuracies.append(acc)

    if epoch % 5 == 0 or epoch == 1:
        ms = int((time.time() - t0) * 1000 / max(1, n_steps))
        print(f'Epoch {epoch:>3}/{EPOCHS}  {n_steps}/{n_steps} '
              f'━━━━━━━━━━━━━━━━━━━━  0s {ms}ms/step  '
              f'accuracy: {acc:.4f}  loss: {loss.item():.4f}')

print(f'\n[OK] Done! Accuracy: {accuracies[-1]*100:.2f}% | Loss: {losses[-1]:.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7: กราฟ Accuracy & Loss                           ║
# ╚══════════════════════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
mode_label = 'Real Data' if USE_REAL_DATA else 'Synthetic'
fig.suptitle(f'Savannakhet GNN — Training Results ({mode_label}: {NUM_USERS} users x {NUM_PLACES} places)',
             fontsize=12, fontweight='bold')
ep = range(1, EPOCHS + 1)

ax1.plot(ep, accuracies, color='#2563EB', lw=2, label='Accuracy')
ax1.fill_between(ep, accuracies, alpha=0.15, color='#2563EB')
ax1.axhline(0.85, color='#059669', ls='--', lw=1.5, label='Target 85%')
ax1.set(title='Model Accuracy', xlabel='Epochs', ylabel='Accuracy', ylim=(0.4, 1.05))
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, losses, color='#DC2626', lw=2, label='Loss')
ax2.fill_between(ep, losses, alpha=0.12, color='#DC2626')
ax2.set(title='Model Loss', xlabel='Epochs', ylabel='Loss', ylim=(0, None))
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Saved: /content/training_results.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8: ทดสอบ Recommendation                           ║
# ╚══════════════════════════════════════════════════════════╝

def recommend(username: str, top_k: int = 5):
    model.eval()
    user_rec = next((u for u in USERS_DATA if u.get('username') == username), None)
    if user_rec is None:
        print(f'[!] ไม่พบ user: {username}')
        print(f'    Users: {[u.get("username") for u in USERS_DATA[:5]]}...')
        return

    u_idx = user_id2idx[user_rec['id']]
    prefs = user_rec.get('preferences', [])

    with torch.no_grad():
        embs   = model(data.x_dict, data.edge_index_dict)
        u_emb  = embs['user'][u_idx]
        p_embs = embs['place']

        gnn_scores = F.cosine_similarity(p_embs, u_emb.unsqueeze(0))

        u_pref_vec = data['user'].x[u_idx, :len(CATEGORIES)]
        p_cat_vecs = data['place'].x[:, :len(CATEGORIES)]
        cb_scores  = (F.cosine_similarity(p_cat_vecs, u_pref_vec.unsqueeze(0))
                      if u_pref_vec.sum() > 0 else torch.zeros(NUM_PLACES).to(device))

        scores = 0.6 * gnn_scores + 0.4 * cb_scores
        top_vals, top_idxs = torch.topk(scores, k=min(top_k, NUM_PLACES))

    print('=' * 65)
    print(f'  Top {top_k} Recommendations for: {username}')
    print(f'  Preferences: {prefs or "(none)"}')
    print('=' * 65)
    for rank, (val, idx) in enumerate(zip(top_vals, top_idxs), 1):
        p   = PLACES_DATA[idx.item()]
        pct = max(0, min(100, (val.item() + 1) / 2 * 100))
        gnn = max(0, min(100, (gnn_scores[idx].item() + 1) / 2 * 100))
        cb  = max(0, min(100, (cb_scores[idx].item() + 1) / 2 * 100))
        mrk = 'v' if p.get('category') in prefs else ' '
        print(f'  {rank}.[{mrk}] {str(p["name"]):<30} [{p.get("category","?")}] Rating:{p.get("rating",0)}')
        print(f'       Score:{pct:.1f}% | GNN(60%):{gnn:.1f}% | CB(40%):{cb:.1f}%')
    print()

# ── ทดสอบ 3 users แรก ──
for u in USERS_DATA[:3]:
    recommend(u['username'], top_k=5)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9: บันทึกโมเดล + Download                         ║
# ╚══════════════════════════════════════════════════════════╝

suffix    = 'real' if USE_REAL_DATA else 'synthetic'
SAVE_PATH = f'/content/gnn_savannakhet_{suffix}.pt'

torch.save({
    'epoch'        : EPOCHS,
    'model_state'  : model.state_dict(),
    'final_acc'    : accuracies[-1],
    'final_loss'   : losses[-1],
    'user_id2idx'  : user_id2idx,
    'place_id2idx' : place_id2idx,
    'place_idx2id' : place_idx2id,
    'metadata': {
        'hidden_dim' : 64,
        'categories' : CATEGORIES,
        'num_users'  : NUM_USERS,
        'num_places' : NUM_PLACES,
        'source'     : suffix,
    }
}, SAVE_PATH)

print('\n' + '=' * 65)
print('  Training Summary')
print('=' * 65)
print(f'  Mode         : {"Real Data" if USE_REAL_DATA else "Synthetic"}')
print(f'  Architecture : HeteroGraphSAGE (K=2, hidden=64)')
print(f'  Dataset      : {NUM_USERS} users x {NUM_PLACES} places')
print(f'  Edges        : {data["user", "interacts_with", "place"].edge_index.shape[1]}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Best Acc     : {max(accuracies)*100:.2f}%')
print(f'  Final Acc    : {accuracies[-1]*100:.2f}%')
print(f'  Final Loss   : {losses[-1]:.4f}')
print(f'  Saved        : {SAVE_PATH}')
print('=' * 65)

from google.colab import files
files.download(SAVE_PATH)
files.download('/content/training_results.png')
print('\n[OK] Downloading files...')
if USE_REAL_DATA:
    print('     -> นำ .pt ไปวางที่ backend/gnn_model.pt เพื่อใช้ใน Production!')